# Sensitivity and counterfactual analysis

This notebook evaluates how fitted model predictions change when baseline input assumptions are modified. These are model-behaviour analyses, not causal effects.

In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from pathlib import Path
import yaml
import pandas as pd
from IPython.display import display

from icare_risk.data.audit import load_admission_file
from icare_risk.models.training import encode_binary_target
import joblib
from icare_risk.explainability import (
    one_at_a_time_sensitivity, evaluate_counterfactuals
)

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

df = load_admission_file(config['outputs']['processed_dataset'])
target = config['columns']['target']
mapping = config.get('target_mapping', config.get('analysis', {}).get('target_mapping', {}))
df['_target_numeric'] = encode_binary_target(
    df[target],
    positive_values=mapping.get('positive_values'),
    negative_values=mapping.get('negative_values'),
)
test = df.loc[df['split'].eq('test') & df['_target_numeric'].notna()].copy()
print(f'Test rows available: {len(test):,}')

Test rows available: 2,483


In [14]:
model_directory = Path(config['outputs']['model_directory'])
model_names = [m['name'] for m in config.get('models', [])]
model_name = model_names[0]
model_name = "lr_clinical_baseline"
bundle = joblib.load(model_directory / f'{model_name}.joblib')
model = bundle['model'] if isinstance(bundle, dict) and 'model' in bundle else bundle
features = bundle.get('features') if isinstance(bundle, dict) else None
if features is None:
    raise KeyError('Saved model bundle must contain the training feature list.')
X_test = test[features].copy()
print(f'Using model: {model_name}')
print(f'Features: {len(features)}')

Using model: lr_clinical_baseline
Features: 16


## One-at-a-time sensitivity

Edit the scenarios below using clinically meaningful values available at the prediction point.

In [15]:
scenarios = {
    'AGE_AT_ADMISSION': [50, 65, 80],
    # 'kim_hx_prior_esbl': [0, 1],
    # 'charlson_score': [0, 3, 6],
}
available = {feature: values for feature, values in scenarios.items() if feature in X_test.columns}
missing = sorted(set(scenarios) - set(available))
if missing:
    print('Skipped missing features:', missing)
sensitivity_table = one_at_a_time_sensitivity(model, X_test, available) if available else pd.DataFrame()
display(sensitivity_table)


,feature,scenario,value,mean_probability,mean_change,median_change
0,AGE_AT_ADMISSION,baseline,NaN,0.496892,0.000000,NaN
1,AGE_AT_ADMISSION,AGE_AT_ADMISSION=50,50.0,0.499368,0.002477,0.002195
2,AGE_AT_ADMISSION,AGE_AT_ADMISSION=65,65.0,0.493918,-0.002974,-0.003155
3,AGE_AT_ADMISSION,AGE_AT_ADMISSION=80,80.0,0.488468,-0.008424,-0.008413


## Named counterfactual scenarios

Each scenario modifies multiple input variables simultaneously. Interpret results as changes in model output under hypothetical inputs, not as treatment effects.

In [22]:
counterfactuals = {
    "older_patient": {
        "AGE_AT_ADMISSION": 80,
    },
    "younger_patient": {
        "AGE_AT_ADMISSION": 50,
    },
}
valid_counterfactuals = {
    name: changes
    for name, changes in counterfactuals.items()
    if all(feature in X_test.columns for feature in changes)
}

print("Model:", model_name)
print("Valid scenarios:", valid_counterfactuals)

counterfactual_table = evaluate_counterfactuals(
    model,
    X_test,
    valid_counterfactuals,
)

display(counterfactual_table)


Model: lr_clinical_baseline
Valid scenarios: {'older_patient': {'AGE_AT_ADMISSION': 80}, 'younger_patient': {'AGE_AT_ADMISSION': 50}}


,scenario,n_modified_features,mean_probability,mean_change,median_change
0,baseline,0,0.496892,0.000000,0.000000
1,older_patient,1,0.488468,-0.008424,-0.008413
2,younger_patient,1,0.499368,0.002477,0.002195


In [23]:
output_dir = Path(config['outputs'].get('sensitivity_directory', 'results/sensitivity'))
output_dir.mkdir(parents=True, exist_ok=True)
if not sensitivity_table.empty:
    sensitivity_table.to_csv(output_dir / f'{model_name}_one_at_a_time.csv', index=False)
if not counterfactual_table.empty:
    counterfactual_table.to_csv(output_dir / f'{model_name}_counterfactuals.csv', index=False)
print(f'Saved results to {output_dir}')

Saved results to results/sensitivity
